In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/data.csv')


print("Initial Data Info:")
df.info()

print("\nMissing values before cleaning:")
display(df.isnull().sum())

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5005 entries, 0 to 5004
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order No           5005 non-null   object 
 1   Order Date         5005 non-null   object 
 2   Customer Name      5005 non-null   object 
 3   Address            5004 non-null   object 
 4   City               5005 non-null   object 
 5   State              5005 non-null   object 
 6   Customer Type      5005 non-null   object 
 7   Account Manager    5005 non-null   object 
 8   Order Priority     5005 non-null   object 
 9   Product Name       5005 non-null   object 
 10  Product Category   5005 non-null   object 
 11  Product Container  5005 non-null   object 
 12  Ship Mode          5005 non-null   object 
 13  Ship Date          5005 non-null   object 
 14  Cost Price         5005 non-null   object 
 15  Retail Price       5005 non-null   object 
 16  Profi

,0
Order No,0
Order Date,0
Customer Name,0
Address,1
City,0
State,0
Customer Type,0
Account Manager,0
Order Priority,0
Product Name,0


In [40]:
duplicate_rows = df[df.duplicated()]
print(f"Number of duplicate rows found: {len(duplicate_rows)}")

df_cleaned = df.drop_duplicates().copy()
print(f"Number of rows after removing duplicates: {len(df_cleaned)}")

Number of duplicate rows found: 5
Number of rows after removing duplicates: 5000


In [41]:
mean_order_quantity = df_cleaned['Order Quantity'].mean()
df_cleaned['Order Quantity'].fillna(mean_order_quantity, inplace=True)
print(f"Missing 'Order Quantity' values filled with mean: {mean_order_quantity:.2f}")

mode_address = df_cleaned['Address'].mode()[0]
df_cleaned['Address'].fillna(mode_address, inplace=True)
print(f"Missing 'Address' values filled with mode: {mode_address}")

print("\nMissing values after handling Order Quantity and Address:")
display(df_cleaned.isnull().sum())

Missing 'Order Quantity' values filled with mean: 26.48
Missing 'Address' values filled with mode: 180 High Street,Windsor

Missing values after handling Order Quantity and Address:


/tmp/ipykernel_16750/37543226.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned['Order Quantity'].fillna(mean_order_quantity, inplace=True)
/tmp/ipykernel_16750/37543226.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, 

,0
Order No,0
Order Date,0
Customer Name,0
Address,0
City,0
State,0
Customer Type,0
Account Manager,0
Order Priority,0
Product Name,0


In [44]:
df_cleaned['Ship Date'] = pd.to_datetime(df_cleaned['Ship Date'])
print("Data type of 'Order Date' column after conversion:")
print(df_cleaned['Order Date'].dtype)
print("Data type of 'Ship Date' column after conversion:")
print(df_cleaned['Ship Date'].dtype)

# Function to clean and convert currency strings to numeric
def clean_currency(s):
    if isinstance(s, str):
        s = s.replace('$', '').replace(',', '').replace('%', '')
        try:
            return float(s)
        except ValueError:
            return np.nan
    return s

# Apply cleaning and conversion to currency columns
currency_cols = ['Cost Price', 'Retail Price', 'Profit Margin', 'Sub Total', 'Discount %', 'Discount $', 'Order Total', 'Shipping Cost', 'Total']
for col in currency_cols:
    df_cleaned[col] = df_cleaned[col].apply(clean_currency)

# Fill any NaNs that might have been introduced during conversion with 0 or a suitable value
# For simplicity, we'll fill with 0 here, or you could re-evaluate specific columns if needed
for col in currency_cols:
    df_cleaned[col].fillna(0, inplace=True)

print("\nData types of currency columns after conversion:")
display(df_cleaned[currency_cols].dtypes)

Data type of 'Order Date' column after conversion:
datetime64[ns]
Data type of 'Ship Date' column after conversion:
datetime64[ns]

Data types of currency columns after conversion:


/tmp/ipykernel_16750/2066980168.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(0, inplace=True)


,0
Cost Price,float64
Retail Price,float64
Profit Margin,float64
Sub Total,float64
Discount %,float64
Discount $,float64
Order Total,float64
Shipping Cost,float64
Total,float64


In [45]:
df_cleaned['Product Category'] = df_cleaned['Product Category'].str.lower()

print("Unique values in 'Product Category' after standardization:")
print(df_cleaned['Product Category'].unique())



Unique values in 'Product Category' after standardization:
['office supplies' 'technology' 'furniture']


### Step 5: Handle Outliers in 'Total Amount'

We will use the Interquartile Range (IQR) method to identify and remove outliers from the 'Total Amount' column.

In [46]:
Q1 = df_cleaned['Total'].quantile(0.25)
Q3 = df_cleaned['Total'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
df_cleaned = df_cleaned[(df_cleaned['Total'] >= lower_bound) & (df_cleaned['Total'] <= upper_bound)]

print(f"Outlier bounds for 'Total': Lower = {lower_bound:.2f}, Upper = {upper_bound:.2f}")
print(f"Number of rows after outlier removal: {len(df_cleaned)}")

Outlier bounds for 'Total': Lower = -452.00, Upper = 922.55
Number of rows after outlier removal: 4295


In [47]:
print("Final Data Info after Cleaning:")
df_cleaned.info()

print("\nFirst 5 rows of the cleaned DataFrame:")
display(df_cleaned.head())

Final Data Info after Cleaning:
<class 'pandas.core.frame.DataFrame'>
Index: 4295 entries, 1 to 4999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Order No           4295 non-null   object        
 1   Order Date         4295 non-null   datetime64[ns]
 2   Customer Name      4295 non-null   object        
 3   Address            4295 non-null   object        
 4   City               4295 non-null   object        
 5   State              4295 non-null   object        
 6   Customer Type      4295 non-null   object        
 7   Account Manager    4295 non-null   object        
 8   Order Priority     4295 non-null   object        
 9   Product Name       4295 non-null   object        
 10  Product Category   4295 non-null   object        
 11  Product Container  4295 non-null   object        
 12  Ship Mode          4295 non-null   object        
 13  Ship Date          4295 non-null   d

,Order No,Order Date,Customer Name,Address,City,State,Customer Type,Account Manager,Order Priority,Product Name,...,Cost Price,Retail Price,Profit Margin,Order Quantity,Sub Total,Discount %,Discount $,Order Total,Shipping Cost,Total
1,5001-1,2015-10-24,Shahid Hopkins,"438 Victoria Avenue,Chatswood",Sydney,NSW,Corporate,Natasha Song,Medium,Bagged Rubber Bands,...,0.24,1.26,1.02,8.0,45.20,3.0,0.00,45.90,0.70,46.91
2,5004-1,2014-03-13,Dennis Pardue,"412 Brunswick St,Fitzroy",Melbourne,VIC,Consumer,Connor Betts,Not Specified,TechSavi Cordless Navigator Duo,...,42.11,80.98,38.87,45.0,873.32,4.0,72.23,837.57,7.18,82.58
3,5009-1,2013-02-18,Sean Wendt,"145 Ramsay St,Haberfield",Sydney,NSW,Small Business,Phoebe Gour,Critical,Artisan Printable Repositionable Plastic Tabs,...,5.33,8.60,3.27,16.0,73.52,1.0,4.35,740.67,6.19,730.92
4,5010-1,2014-09-13,Christina Vanderzanden,"188 Pitt Street,Sydney",Sydney,NSW,Small Business,Tina Carlton,Not Specified,Pizazz Drawing Pencil Set,...,1.53,2.78,1.25,49.0,138.46,7.0,5.95,123.77,1.34,125.97
5,5011-1,2013-11-24,Patrick OBrill,"53 Riley Street,Woolloomooloo",Sydney,NSW,Home Office,Tina Carlton,Not Specified,"Alto Parchment Paper, Assorted Colors",...,4.59,7.28,2.69,45.0,197.36,8.0,12.98,183.58,11.15,189.43


In [48]:
df_cleaned.to_csv('cleaned_data.csv', index=False)
print("Cleaned data saved to 'cleaned_data.csv'")

Cleaned data saved to 'cleaned_data.csv'
